In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, when, to_timestamp, lit, max as spark_max, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
spark = SparkSession.builder \
    .appName("Weather Streaming ETL + Alerts") \
    .config("spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.driver.extraClassPath",
            "/home/jovyan/work/Graduation/mssql-jdbc-13.2.1.jre8.jar") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session started successfully.")
KAFKA_TOPIC = "weather_s" 
JDBC_URL = "jdbc:sqlserver://host.docker.internal:1433;databaseName=iot_db1;encrypt=false;"
CONNECTION_PROPERTIES = {
    "user": "sa",
    "password": "123456789",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
schema = StructType([
    StructField("measurement_id", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("location_id", StringType(), True),
    StructField("location_name", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", IntegerType(), True),
    StructField("pressure", IntegerType(), True),
    StructField("air_quality", IntegerType(), True),
    StructField("noise_level", IntegerType(), True), 
    StructField("battery_level", IntegerType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("wind_direction", StringType(), True),
    StructField("rain_level_mm", DoubleType(), True),
    StructField("visibility_km", DoubleType(), True),
    StructField("uv_index", IntegerType(), True),
    StructField("cloud_coverage_percent", IntegerType(), True),
    StructField("season", StringType(), True),
    StructField("day_period", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("temperature_status", StringType(), True),
    StructField("battery_status", StringType(), True),
    StructField("anomaly_flag", IntegerType(), True)
])
raw_df = spark.readStream.format("kafka") \
   .option("kafka.bootstrap.servers", "kafka:9092") \
   .option("subscribe", KAFKA_TOPIC) \
   .option("startingOffsets", "latest") \
   .load()
df_parsed = raw_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")
TIME_FORMAT = "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"
df_with_correct_ts = df_parsed.withColumn(
    "timestamp", 
    to_timestamp(col("timestamp"), TIME_FORMAT)
)
raw_values_df = df_with_correct_ts \
    .withColumn("temperature_status", lit(None).cast(StringType())) \
    .withColumn("battery_status", lit(None).cast(StringType())) \
    .withColumn("anomaly_flag", lit(None).cast(IntegerType()))
df = df_with_correct_ts
df = df.withColumn(
    "temperature_status",
    when(col("temperature") > 30, "High")
    .when(col("temperature") < 22, "Low")
    .otherwise("Normal")
)
df = df.withColumn(
    "battery_status",
    when(col("battery_level") < 20, "Low Battery")
    .otherwise("OK")
)
df = df.withColumn(
    "anomaly_flag",
    when((col("temperature") > 45) | (col("temperature") < 5), 1) 
    .otherwise(0)
)
alerts_df = df.select(
    "measurement_id", "device_id", "location_name", "timestamp", "temperature", "humidity",
    when(col("temperature") > 40, lit("Extreme High Temperature"))
    .when(col("air_quality") > 90, lit("Poor Air Quality (AQI > 90)"))
    .when(col("uv_index") > 8, lit("High UV Index Warning"))
    .when(col("wind_speed") > 15, lit("High Wind Speed Warning"))
    .when(col("battery_level") < 20, lit("CRITICAL: Low Battery Level"))
    .when(col("noise_level") > 85, lit("High Noise Pollution Alert (dB > 85)"))
    .otherwise(None).alias("alert_message")
).filter(col("alert_message").isNotNull())
def write_raw_data_to_sql(batch_df, batch_id):
    print(f"Writing raw data batch {batch_id} to iot_raw_data_s...")
    batch_df.write.jdbc(
        url=JDBC_URL,
        table="iot_raw_data_s",
        mode="append",
        properties=CONNECTION_PROPERTIES
    )
def write_transformed_data_to_sql(batch_df, batch_id):
    print(f"Writing transformed data batch {batch_id} to iot_data_s...")
    batch_df.write.jdbc(
        url=JDBC_URL,
        table="iot_data_s",
        mode="append",
        properties=CONNECTION_PROPERTIES
    )

def write_alerts_to_sql(batch_df, batch_id):
    print(f"Writing alerts batch {batch_id} to iot_alerts_s...")
    batch_df.write.jdbc(
        url=JDBC_URL,
        table="iot_alerts_s",
        mode="append",
        properties=CONNECTION_PROPERTIES
    )
processed_to_kafka = df.selectExpr("to_json(struct(*)) AS value") \
    .writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "weather_processed_s") \
    .option("checkpointLocation", "/tmp/checkpoints/weather_processed_s") \
    .outputMode("append") \
    .start()
query_raw = raw_values_df.writeStream \
    .foreachBatch(write_raw_data_to_sql) \
    .outputMode("append") \
    .queryName("RawValuesWriter") \
    .start()
query_data = df.writeStream \
    .foreachBatch(write_transformed_data_to_sql) \
    .outputMode("append") \
    .queryName("TransformedDataWriter") \
    .start()
query_alerts = alerts_df.writeStream \
    .foreachBatch(write_alerts_to_sql) \
    .outputMode("append") \
    .queryName("AlertsWriter") \
    .start()
def send_latest_temp_to_kafka(batch_df, batch_id):
    latest_ts = batch_df.groupBy("device_id").agg(
        spark_max("timestamp").alias("latest_ts")
    )
    latest_readings = batch_df.alias("d").join(
        latest_ts.alias("t"),
        (col("d.device_id") == col("t.device_id")) & (col("d.timestamp") == col("t.latest_ts")),
        "inner"
    ).select("d.device_id", "d.temperature", "d.timestamp")
    
    latest_readings.select(to_json(struct("device_id", "temperature", "timestamp")).alias("value")) \
        .write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "kafka:9092") \
        .option("topic", "device_temp_latest_s") \
        .save()
latest_to_kafka = df.writeStream \
    .foreachBatch(send_latest_temp_to_kafka) \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/checkpoints/device_temp_latest_s") \
    .start() 
print("All streaming queries started. Waiting for termination...")
query_raw.awaitTermination()
query_data.awaitTermination()
query_alerts.awaitTermination()

Spark Session started successfully.


IllegalArgumentException: Cannot start query with name RawValuesWriter as a query with that name is already active in this SparkSession

Writing transformed data batch 449 to iot_data_s...
Writing alerts batch 449 to iot_alerts_s...
Writing raw data batch 449 to iot_raw_data_s...
Writing alerts batch 450 to iot_alerts_s...
Writing raw data batch 450 to iot_raw_data_s...
Writing transformed data batch 450 to iot_data_s...
Writing raw data batch 451 to iot_raw_data_s...
Writing transformed data batch 451 to iot_data_s...
Writing alerts batch 451 to iot_alerts_s...
Writing raw data batch 452 to iot_raw_data_s...Writing transformed data batch 452 to iot_data_s...

Writing alerts batch 452 to iot_alerts_s...
Writing raw data batch 453 to iot_raw_data_s...
Writing transformed data batch 453 to iot_data_s...
Writing alerts batch 453 to iot_alerts_s...
Writing raw data batch 454 to iot_raw_data_s...
Writing alerts batch 454 to iot_alerts_s...
Writing transformed data batch 454 to iot_data_s...
Writing raw data batch 455 to iot_raw_data_s...
Writing transformed data batch 455 to iot_data_s...
Writing alerts batch 455 to iot_aler